<a href="https://colab.research.google.com/github/maha-naveed77/Flyrank-Internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maha-naveed77/Flyrank-Internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [ ]:
# Method: Random Forest, compared against Logistic Regression as a simpler baseline-of-the-model-menu. Random Forest fits this lane because refresh-worthiness likely depends on combinations of signals (visibility + position + engagement) rather than any single linear relationship — the w04 signal audit already showed staleness alone was a MIXED, non-monotonic signal, which suggests interactions matter more than raw linear weighting. I'm not using clustering since this is a scoring/ranking task with a clear proxy label, not an unsupervised grouping question.


In [1]:
# setup (new notebook — nothing carries over)
import duckdb, pandas as pd, numpy as np
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute("INSTALL httpfs;"); con.execute("LOAD httpfs;")
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}');")
BASE = "hf://datasets/FlyRank/internship-warehouse"

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [ ]:
# Grouped split by client_hash_id — clients, not individual pages, are randomly assigned to train/test, so no client's pages leak across both sets (matching the starter pipeline's own client_holdout strategy). This is the honest choice because pages from the same client share systemic quirks (as seen in w04 — a handful of clients dominated the whole top-20), so a page-level random split would let the model memorize client-specific patterns instead of learning generalizable signal.

In [2]:
# Build features from April (month=2026-04), label from actual May outcome (month=2026-05)
q_apr = f"""
SELECT client_hash_id, content_hash_id,
       SUM(gsc_impressions) AS impressions_apr,
       SUM(gsc_clicks) AS clicks_apr,
       SUM(gsc_sum_position) AS sum_position_apr,
       SUM(sessions_ai) AS sessions_ai_apr,
       SUM(scroll_events) AS scroll_events_apr
FROM read_parquet('{BASE}/fact_content_daily_performance/*/*.parquet')
WHERE month = '2026-04'
GROUP BY 1,2
HAVING SUM(gsc_impressions) > 0
"""
feat_apr = con.execute(q_apr).df()
feat_apr['avg_position_apr'] = feat_apr['sum_position_apr'] / feat_apr['impressions_apr']
feat_apr['ctr_apr'] = feat_apr['clicks_apr'] / feat_apr['impressions_apr']

q_may = f"""
SELECT client_hash_id, content_hash_id,
       SUM(gsc_impressions) AS impressions_may
FROM read_parquet('{BASE}/fact_content_daily_performance/*/*.parquet')
WHERE month = '2026-05'
GROUP BY 1,2
"""
label_may = con.execute(q_may).df()

df = feat_apr.merge(label_may, on=['client_hash_id','content_hash_id'], how='left')
df['impressions_may'] = df['impressions_may'].fillna(0)
df['is_declining'] = (df['impressions_may'] < df['impressions_apr'] * 0.8).astype(int)  # >20% drop = decline

print(df['is_declining'].value_counts(normalize=True))

# grouped split by client
clients = df['client_hash_id'].unique()
np.random.seed(42)
test_clients = np.random.choice(clients, size=int(len(clients)*0.2), replace=False)
train_df = df[~df['client_hash_id'].isin(test_clients)]
test_df = df[df['client_hash_id'].isin(test_clients)]
print(f"Train: {len(train_df)} rows, {train_df['client_hash_id'].nunique()} clients")
print(f"Test:  {len(test_df)} rows, {test_df['client_hash_id'].nunique()} clients")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

is_declining
1    0.503009
0    0.496991
Name: proportion, dtype: float64
Train: 138437 rows, 41 clients
Test:  56323 rows, 10 clients


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [4]:
train_df = train_df.copy()
train_df['position_tier'] = pd.cut(train_df['avg_position_apr'], bins=bins, labels=labels_).astype(str)
test_df['position_tier'] = pd.cut(test_df['avg_position_apr'], bins=bins, labels=labels_).astype(str)

tier_mean_ctr = train_df.groupby('position_tier')['ctr_apr'].mean()

test_df['expected_ctr'] = test_df['position_tier'].map(tier_mean_ctr).astype(float)
test_df['ctr_apr'] = test_df['ctr_apr'].astype(float)
test_df['impressions_apr'] = test_df['impressions_apr'].astype(float)

test_df['baseline_score'] = (test_df['expected_ctr'] - test_df['ctr_apr']) * test_df['impressions_apr']

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [5]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(rf, X_test, y_test, n_repeats=10, random_state=42)
importance_df = pd.DataFrame({'feature': features, 'importance': perm.importances_mean}).sort_values('importance', ascending=False)
print(importance_df)

# look at false positives/negatives among the model's top 50
top50 = test_df.sort_values('rf_score', ascending=False).head(50)
print(f"Of the model's top 50, {top50['is_declining'].sum()} were truly declining, {50 - top50['is_declining'].sum()} were false positives.")
false_pos = top50[top50['is_declining'] == 0]
print(false_pos[['impressions_apr','ctr_apr','avg_position_apr']].describe())

             feature  importance
3            ctr_apr    0.023685
2   avg_position_apr    0.022623
0    impressions_apr    0.019637
1         clicks_apr    0.005449
5  scroll_events_apr    0.002035
4    sessions_ai_apr   -0.000486
Of the model's top 50, 45 were truly declining, 5 were false positives.
       impressions_apr   ctr_apr  avg_position_apr
count         5.000000  5.000000          5.000000
mean        128.600000  0.000336          6.374091
std         260.737991  0.000752          2.543565
min           8.000000  0.000000          2.715966
25%          12.000000  0.000000          5.846154
50%          13.000000  0.000000          5.933333
75%          15.000000  0.000000          7.875000
max         595.000000  0.001681          9.500000


In [ ]:
#Label balance: 50.3% declining vs. 49.7% not — a clean, well-balanced label, no skew problem to worry about.
#Split: 138,437 train rows across 41 clients, 56,323 test rows across 10 clients — a real grouped holdout, no client overlap.
#Feature importance: ctr_apr (0.024) and avg_position_apr (0.023) are the top two drivers, with impressions_apr close behind (0.020). sessions_ai_apr is essentially useless (slightly negative, meaning it's pure noise the model would do just as well without) — consistent with what you found in w03: GA4/session data has very low availability (~4%), so it can't carry much signal.
#Error analysis: of the top 50, 45 were true positives and only 5 were false positives — a strong precision if that holds up against the baseline. Looking at the 5 false positives: they have very low CTR (mean 0.03%) and low impressions (mostly 8–15, one outlier at 595) at worse average positions (mean ~6.4) — these look like small, low-visibility pages the model flagged as "declining" mostly because they're already so small there's little room to fall further, not because they show a real meaningful decline pattern. That's a useful, honest thing to write in your interpretation: the model may slightly over-flag already-tiny pages where the signal is more noise than decline.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.